# 活跃发言人检测 - 算法调优

本笔记本用于交互式调优检测算法的参数。

## 目标

1. 可视化 MAR 时间序列
2. 分析方差曲线
3. 测试不同参数组合
4. 评估误报和漏报率

In [ ]:
# 导入必要的库
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 添加 src 目录到路径
sys.path.insert(0, str(Path.cwd().parent / "src"))

from detector.active_speaker import ActiveSpeakerDetector
from detector.velocity_tracker import VelocityTracker

%matplotlib inline

## 1. 可视化 MAR 时间序列

从摄像头捕获一段视频，绘制 MAR 变化曲线。

In [ ]:
def record_mar_sequence(duration_seconds=10, camera_id=0):
    """
    录制一段时间的 MAR 序列
    
    Args:
        duration_seconds: 录制时长（秒）
        camera_id: 摄像头 ID
    
    Returns:
        mar_sequence: MAR 值列表
    """
    detector = ActiveSpeakerDetector()
    cap = cv2.VideoCapture(camera_id)
    
    mar_sequence = []
    start_time = cv2.getTickCount()
    
    print(f"Recording for {duration_seconds} seconds...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        is_speaking, debug_info = detector.process_frame(frame)
        
        if debug_info['mar'] is not None:
            mar_sequence.append(debug_info['mar'])
        
        # 显示当前帧
        display_frame = detector.draw_debug_info(frame, is_speaking, debug_info)
        cv2.imshow('Recording', display_frame)
        
        # 检查是否超时
        elapsed = (cv2.getTickCount() - start_time) / cv2.getTickFrequency()
        if elapsed >= duration_seconds:
            break
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    detector.close()
    
    print(f"Recorded {len(mar_sequence)} frames")
    return mar_sequence

In [ ]:
# 录制 MAR 序列（请在说话、打哈欠、微笑等不同场景下录制）
mar_seq = record_mar_sequence(duration_seconds=10)

In [ ]:
# 可视化 MAR 时间序列
plt.figure(figsize=(15, 5))
plt.plot(mar_seq, label='MAR', linewidth=1)
plt.xlabel('Frame')
plt.ylabel('MAR Value')
plt.title('MAR Time Series')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. 方差分析

计算滑动窗口方差，可视化方差曲线。

In [ ]:
def calculate_sliding_variance(mar_seq, window_size=15):
    """
    计算滑动窗口方差
    
    Args:
        mar_seq: MAR 序列
        window_size: 窗口大小
    
    Returns:
        variances: 方差序列
    """
    variances = []
    for i in range(len(mar_seq)):
        if i < window_size - 1:
            variances.append(0.0)
        else:
            window = mar_seq[i - window_size + 1 : i + 1]
            variances.append(np.var(window))
    return variances

In [ ]:
# 计算方差
variances = calculate_sliding_variance(mar_seq, window_size=15)

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# MAR 曲线
axes[0].plot(mar_seq, label='MAR', linewidth=1)
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('MAR Value')
axes[0].set_title('MAR Time Series')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 方差曲线
axes[1].plot(variances, label='Variance', color='orange', linewidth=1)
axes[1].axhline(y=0.001, color='r', linestyle='--', label='Threshold (0.001)')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Variance')
axes[1].set_title('Sliding Window Variance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 参数扫描

测试不同的方差阈值和窗口大小组合。

In [ ]:
def test_parameters(mar_seq, variance_thresholds, window_sizes):
    """
    测试不同参数组合
    
    Args:
        mar_seq: MAR 序列
        variance_thresholds: 方差阈值列表
        window_sizes: 窗口大小列表
    
    Returns:
        results: 结果字典
    """
    results = {}
    
    for threshold in variance_thresholds:
        for window_size in window_sizes:
            tracker = VelocityTracker(
                window_size=window_size,
                variance_threshold=threshold,
                min_speaking_frames=9
            )
            
            speaking_frames = 0
            for mar in mar_seq:
                is_speaking, _ = tracker.update(mar)
                if is_speaking:
                    speaking_frames += 1
            
            speaking_ratio = speaking_frames / len(mar_seq)
            results[(threshold, window_size)] = speaking_ratio
    
    return results

In [ ]:
# 定义参数范围
variance_thresholds = [0.0005, 0.001, 0.002, 0.005]
window_sizes = [10, 15, 20, 25]

# 测试参数
results = test_parameters(mar_seq, variance_thresholds, window_sizes)

# 可视化结果
import pandas as pd

# 转换为 DataFrame
data = []
for (threshold, window_size), ratio in results.items():
    data.append({
        'Variance Threshold': threshold,
        'Window Size': window_size,
        'Speaking Ratio': ratio
    })

df = pd.DataFrame(data)
pivot_table = df.pivot(index='Variance Threshold', 
                       columns='Window Size', 
                       values='Speaking Ratio')

print("Speaking Ratio for Different Parameters:")
print(pivot_table)

# 热力图
plt.figure(figsize=(10, 6))
plt.imshow(pivot_table, cmap='viridis', aspect='auto')
plt.colorbar(label='Speaking Ratio')
plt.xlabel('Window Size')
plt.ylabel('Variance Threshold')
plt.title('Parameter Sensitivity Analysis')
plt.xticks(range(len(window_sizes)), window_sizes)
plt.yticks(range(len(variance_thresholds)), variance_thresholds)
plt.show()

## 4. 实时调试

使用滑块交互式调整参数。

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def plot_with_threshold(variance_threshold=0.001, min_speaking_frames=9):
    """
    交互式可视化不同阈值下的检测结果
    """
    tracker = VelocityTracker(
        window_size=15,
        variance_threshold=variance_threshold,
        min_speaking_frames=min_speaking_frames
    )
    
    speaking_status = []
    for mar in mar_seq:
        is_speaking, _ = tracker.update(mar)
        speaking_status.append(1 if is_speaking else 0)
    
    variances = calculate_sliding_variance(mar_seq, window_size=15)
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 10))
    
    # MAR
    axes[0].plot(mar_seq, label='MAR', linewidth=1)
    axes[0].set_ylabel('MAR')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Variance
    axes[1].plot(variances, label='Variance', color='orange', linewidth=1)
    axes[1].axhline(y=variance_threshold, color='r', linestyle='--', 
                    label=f'Threshold ({variance_threshold})')
    axes[1].set_ylabel('Variance')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Speaking Status
    axes[2].plot(speaking_status, label='Speaking', color='green', linewidth=2)
    axes[2].set_ylabel('Speaking (0/1)')
    axes[2].set_xlabel('Frame')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([-0.1, 1.1])
    
    plt.tight_layout()
    plt.show()
    
    speaking_ratio = sum(speaking_status) / len(speaking_status)
    print(f"Speaking Ratio: {speaking_ratio:.2%}")

# 创建交互式控件
interact(
    plot_with_threshold,
    variance_threshold=FloatSlider(min=0.0001, max=0.01, step=0.0001, value=0.001),
    min_speaking_frames=IntSlider(min=3, max=20, step=1, value=9)
)

## 5. 性能测试

测试检测器的 FPS 性能。

In [ ]:
import time

def benchmark_fps(duration_seconds=10, camera_id=0):
    """
    测试 FPS 性能
    """
    detector = ActiveSpeakerDetector()
    cap = cv2.VideoCapture(camera_id)
    
    frame_count = 0
    start_time = time.time()
    
    print(f"Running benchmark for {duration_seconds} seconds...")
    
    while time.time() - start_time < duration_seconds:
        ret, frame = cap.read()
        if not ret:
            break
        
        detector.process_frame(frame)
        frame_count += 1
    
    elapsed = time.time() - start_time
    fps = frame_count / elapsed
    
    cap.release()
    detector.close()
    
    print(f"Processed {frame_count} frames in {elapsed:.2f} seconds")
    print(f"Average FPS: {fps:.2f}")
    
    return fps

# 运行性能测试
fps = benchmark_fps(duration_seconds=10)

## 总结

使用本笔记本可以：

1. 录制和可视化 MAR 时间序列
2. 分析方差曲线的特征
3. 测试不同参数组合的效果
4. 交互式调整阈值
5. 评估系统性能

根据实际使用场景调整参数，找到最佳的检测效果。